In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import warnings

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
"""
Complete Spatial Hedonic Price Model Analysis
Analyzing impact of street tree canopy on property prices with spatial autocorrelation
"""

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Spatial analysis libraries
try:
    import libpysal
    from libpysal.weights import KNN, DistanceBand
    from esda.moran import Moran
    from spreg import OLS, ML_Lag, ML_Error, GM_Lag, GM_Error
except ImportError:
    print("Installing required spatial libraries...")
    print("Run: pip install libpysal esda spreg")


def load_and_prepare_data(filepath):
    """
    Load your GeoDataFrame with property data
    """

    
    # Load the data
    print(f"\n   Loading data from: {filepath}")
    gdf = gpd.read_file(filepath)
    
    if 'log_price' not in gdf.columns:
        gdf['log_price'] = np.log(gdf['GrossSalePrice'])
    
        # Check for required columns
    required_cols = ['log_price', 'geometry']
    missing = [col for col in required_cols if col not in gdf.columns]
    if missing:
        print(f"\n  Missing required columns: {missing}")
    else:
        print(f" all required columns present")
    
    return gdf


def exploratory_analysis(gdf):
    """
    Comprehensive EDA including distributions, correlations, and spatial patterns
    """

    
    
    key_vars = ['log_price', 'GrossSalePrice', 'AgeAtSale', 'LandArea', 
                'TotalFloorArea', 'water_DIST', 'bus_DIST', 'CBD_DIST']
    print(gdf[key_vars].describe())
    
    # 2.2 Distribution of outcome variable
    print("\n2.2 Outcome Variable Distribution")
    print("-" * 40)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Histogram
    axes[0].hist(gdf['log_price'], bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Log Price')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Log Property Price')
    axes[0].axvline(gdf['log_price'].mean(), color='red', 
                    linestyle='--', label='Mean')    
    axes[0].legend()
    
    # Q-Q plot
    stats.probplot(gdf['log_price'], dist="norm", plot=axes[1])
    axes[1].set_title('Q-Q Plot')
    
    # Box plot
    axes[2].boxplot(gdf['log_price'])
    axes[2].set_ylabel('Log Price')
    axes[2].set_title('Box Plot (Outlier Detection)')
    
    plt.tight_layout()
    plt.savefig('01_outcome_distribution.png', dpi=300, bbox_inches='tight')
    print("  Saved: 01_outcome_distribution.png")
    plt.close()
    
    # 2.3 Correlation analysis
    # All numeric columns
    numeric_cols = gdf.select_dtypes(include=[np.number]).columns.tolist()
    if 'geometry' in numeric_cols:
        numeric_cols.remove('geometry')
    
    # Correlation matrix
    corr_matrix = gdf[numeric_cols].corr()
    
    # Plot correlation heatmap for key variables
    key_vars_extended = ['log_price', 'AgeAtSale', 'LandArea', 'TotalFloorArea',
                         'water_DIST', 'bus_DIST', 'CBD_DIST', 'Median_Income',
                        'RnkIMDNoEm', 'RnkIMDNoIn', 'RnkIMDNoCr', 'RnkIMDNoHo', 
                        'RnkIMDNoHe', 'RnkIMDNoEd', 'RnkIMDNoAc', 'Census_Pop',
                        'DECILE_high', 'DECILE_prim','cycleways_DIST', 'cycle_DENS',
                        'year_2018', 'year_2019'
                         ]
    
    # Add canopy variables
    canopy_vars = [col for col in gdf.columns if col.startswith('canopy_')]
    key_vars_extended.extend(canopy_vars)
    
    # Filter to existing columns
    key_vars_extended = [v for v in key_vars_extended if v in corr_matrix.columns]
    
    plt.figure(figsize=(16, 14))
    sns.heatmap(corr_matrix.loc[key_vars_extended, key_vars_extended], 
                annot=False, cmap='coolwarm', center=0, 
                vmin=-1, vmax=1, square=True)
    plt.title('Correlation Matrix: Key Variables', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('02_correlation_matrix.png', dpi=300, bbox_inches='tight')
    print(" Saved: 02_correlation_matrix.png")
    plt.close()
    
    # Check multicollinearity - IMD variables
    imd_vars = [col for col in gdf.columns if 'RnkIMDNo' in col]
    if len(imd_vars) > 0:
        imd_corr = gdf[imd_vars].corr()
        print(imd_corr)
        print(f"\n Average correlation: {imd_corr.values[np.triu_indices_from(imd_corr.values, k=1)].mean():.3f}")
    
    # Check multicollinearity - Canopy variables
    if len(canopy_vars) > 0:
        canopy_corr = gdf[canopy_vars].corr()
        print(canopy_corr)
        print(f"\n Average correlation: {canopy_corr.values[np.triu_indices_from(canopy_corr.values, k=1)].mean():.3f}")
    
    # 2.4 Spatial distribution
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Property prices
    gdf.plot(column='log_price', cmap='YlOrRd', legend=True, 
             ax=axes[0], markersize=1, alpha=0.6)
    axes[0].set_title('Spatial Distribution: Log Property Price', fontweight='bold')
    axes[0].axis('off')
    
    # Canopy (example: 100-150m band)
    if 'canopy_100_150' in gdf.columns:
        gdf.plot(column='canopy_100_150', cmap='Greens', legend=True, 
                 ax=axes[1], markersize=1, alpha=0.6)
        axes[1].set_title('Spatial Distribution: Canopy (100-150m)', fontweight='bold')
        axes[1].axis('off')
    
    plt.tight_layout()
    plt.savefig('03_spatial_distribution.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    return key_vars_extended, canopy_vars, imd_vars

def feature_engineering(gdf, imd_vars):
    """
    Handle multicollinearity and create aggregated features
    """
    
    gdf_model = gdf.copy()
    
    # 3.0 Handle special codes in DECILE variables
    print("\n3.0 Handling Special Codes in Decile Variables")
    print("-" * 40)
    
    for var in ['DECILE_prim', 'DECILE_high']:
        if var in gdf_model.columns:
            n_special = (gdf_model[var] == 99).sum()
            pct_special = (gdf_model[var] == 99).mean() * 100
            
            if n_special > 0:
                print(f"\n   {var}:")
                print(f"      Special code (99): {n_special} obs ({pct_special:.1f}%)")
                
                # Drop observations with code 99
                gdf_model = gdf_model[gdf_model[var] != 99]
                print(f"     Dropped {n_special} observations with {var} = 99")
    
    # 3.1 Handle IMD multicollinearity - create composite index
    print("\n3.1 Creating Composite Deprivation Index")
    print("-" * 40)
    
    if len(imd_vars) > 0:
        # Standardize IMD variables
        scaler = StandardScaler()
        imd_scaled = scaler.fit_transform(gdf_model[imd_vars])
        
        # Create composite (mean of standardized ranks)
        gdf_model['IMD_composite'] = imd_scaled.mean(axis=1)
        print(f" Created IMD_composite from {len(imd_vars)} deprivation measures")
    
    # 3.2 Aggregate canopy into distance categories
    
    canopy_cols = [col for col in gdf_model.columns if col.startswith('canopy_')]
    
    if len(canopy_cols) > 0:
        # Keep individual bands up to 200m
        keep_individual = ['canopy_0_25', 'canopy_25_50', 'canopy_50_75', 
                          'canopy_75_100', 'canopy_100_150', 'canopy_150_200']
        
        available_individual = [c for c in keep_individual if c in canopy_cols]
        print(f"  Keeping individual bands (0-200m): {', '.join(available_individual)}")
        
        # Aggregate 200-400m
        distant = [c for c in canopy_cols if any(x in c for x in ['200_250', '250_300', '300_350', '350_400'])]
        if distant:
            gdf_model['canopy_200_400'] = gdf_model[distant].sum(axis=1)
            print(f" Created aggregated band: canopy_200_400 (from {len(distant)} bands)")
    
    else:
        print("   No canopy variables found")
    
    # 3.3 Log-transform distance variables
    
    dist_vars = ['water_DIST', 'bus_DIST', 'CBD_DIST', 'cycleways_DIST']
    for var in dist_vars:
        if var in gdf_model.columns:
            # Add small constant to avoid log(0)
            gdf_model[f'log_{var}'] = np.log(gdf_model[var] + 1)
            print(f"   Created log_{var}")
    
    return gdf_model

def calculate_vif(X, var_names):
    """
    Calculate Variance Inflation Factor for each variable
    """
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    
    vif_data = pd.DataFrame()
    vif_data["Variable"] = var_names[1:]  # Skip constant
    vif_data["VIF"] = [variance_inflation_factor(X, i) for i in range(1, X.shape[1])]
    
    return vif_data

def fit_baseline_ols(gdf):
    """
    Fit baseline OLS hedonic model and test for multicollinearity
    """
    print("\n" + "="*80)
    print("STEP 4: BASELINE OLS HEDONIC MODEL")
    print("="*80)
    
    
    # Core hedonic variables
    X_vars = [
        'AgeAtSale', 'LandArea', 'TotalFloorArea',
        'log_water_DIST', 'log_bus_DIST', 'log_CBD_DIST', 'log_cycleways_DIST',
        'Median_Income', 'IMD_composite',
        'canopy_0_25', 'canopy_25_50', 'canopy_50_75', 'canopy_75_100',
        'canopy_100_150', 'canopy_150_200', 'canopy_200_400',
        'year_2018', 'year_2019', 'DECILE_prim', 'DECILE_high'
    ]
    
    print(gdf.columns)
    
    # Filter to existing columns
    X_vars = [v for v in X_vars if v in gdf.columns]
    
    print(f"   Dependent variable: log_price")
    print(f"   Independent variables ({len(X_vars)}):")
    for var in X_vars:
        print(f"      • {var}")
    
    # Prepare data
    y = gdf['log_price'].values.reshape(-1, 1)
    X = gdf[X_vars].values
    
    # Add constant
    X_const = np.hstack([np.ones((X.shape[0], 1)), X])
    var_names = ['Constant'] + X_vars
    
    # 4.2 Test for Multicollinearity (VIF)
    print("\n4.2 Variance Inflation Factor (VIF) Test")
    print("-" * 40)
    print("   Testing for multicollinearity...")
    
    vif_results = calculate_vif(X_const, var_names)
    
    print("\n   VIF Results:")
    print("   " + "-" * 60)
    print(vif_results.to_string(index=False))
    
    print("\n   Interpretation:")
    print("      VIF < 5:   No multicollinearity concern")
    print("      VIF 5-10:  Moderate multicollinearity")
    print("      VIF > 10:  Severe multicollinearity")
    
    high_vif = vif_results[vif_results['VIF'] > 10]
    if len(high_vif) > 0:
        print(f"\nVariables with VIF > 10 (severe multicollinearity):")
        for _, row in high_vif.iterrows():
            print(f"      • {row['Variable']}: VIF = {row['VIF']:.2f}")
    
    moderate_vif = vif_results[(vif_results['VIF'] >= 5) & (vif_results['VIF'] <= 10)]
    if len(moderate_vif) > 0:
        print(f"\n Variables with VIF 5-10 (moderate multicollinearity):")
        for _, row in moderate_vif.iterrows():
            print(f"      • {row['Variable']}: VIF = {row['VIF']:.2f}")
    
    
    ols_model = OLS(y, X_const, name_y='log_price', name_x=var_names)
    
    print("\n" + "="*60)
    print("OLS REGRESSION RESULTS")
    print("="*60)
    print(ols_model.summary)
    
    # Extract residuals for spatial analysis
    residuals = ols_model.u
    
    return ols_model, residuals, X_vars, vif_results


def create_spatial_weights(gdf, method='knn', k=8, distance_threshold=500):
    """
    Create spatial weights matrix
    """
    print("\n" + "="*80)
    print("STEP 5: CONSTRUCTING SPATIAL WEIGHTS MATRIX")
    print("="*80)
    
    # Get coordinates
    coords = np.array(list(zip(gdf.geometry.x, gdf.geometry.y)))
    
    if method == 'knn':
        print(f"\n5.1 K-Nearest Neighbors (k={k})")
        print("-" * 40)
        w = KNN.from_array(coords, k=k)
        print(f" Created KNN weights matrix")
        cardinalities = list(w.cardinalities.values())
        print(f"   • Average neighbors: {np.mean(cardinalities):.2f}")
        
    elif method == 'distance':
        print(f"\n5.1 Distance Band (threshold={distance_threshold}m)")
        print("-" * 40)
        w = DistanceBand.from_array(coords, threshold=distance_threshold)
        print(f"  Created distance-based weights matrix")
        cardinalities = list(w.cardinalities.values())
        print(f"   • Average neighbors: {np.mean(cardinalities):.2f}")
        print(f"   • Min neighbors: {min(cardinalities)}")
        print(f"   • Max neighbors: {max(cardinalities)}")
    
    # Row-standardize
    w.transform = 'r'
    print(f"  Row-standardized weights")
    
    return w


def test_spatial_autocorrelation(residuals, w, gdf):
    """
    Test for spatial autocorrelation in OLS residuals
    """
    print("\n" + "="*80)
    print("STEP 6: TESTING FOR SPATIAL AUTOCORRELATION")
    print("="*80)
    
    # Moran's I test
    print("\n6.1 Global Moran's I Test")
    print("-" * 40)
    
    moran = Moran(residuals.flatten(), w)
    
    print(f"   Moran's I statistic: {moran.I:.4f}")
    print(f"   Expected I: {moran.EI:.4f}")
    print(f"   Variance: {moran.VI_norm:.6f}")
    print(f"   Z-score: {moran.z_norm:.4f}")
    print(f"   P-value: {moran.p_norm:.6f}")
    
    if moran.p_norm < 0.01:
        print(f"\n   *** STRONG spatial autocorrelation detected (p < 0.01)")
        print(f"   *** Spatial model is necessary!")
    elif moran.p_norm < 0.05:
        print(f"\n   ** Significant spatial autocorrelation (p < 0.05)")
    else:
        print(f"\n   No significant spatial autocorrelation detected")
    
    # Map residuals
    print("\n6.2 Mapping OLS Residuals")
    print("-" * 40)
    
    gdf_plot = gdf.copy()
    gdf_plot['ols_residuals'] = residuals
    
    fig, ax = plt.subplots(figsize=(12, 10))
    gdf_plot.plot(column='ols_residuals', cmap='RdBu', legend=True,
                  ax=ax, markersize=2, alpha=0.6,
                  vmin=-residuals.std()*2, vmax=residuals.std()*2)
    ax.set_title('OLS Residuals (Spatial Pattern)', fontweight='bold', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('04_ols_residuals_map.png', dpi=300, bbox_inches='tight')
    print("  Saved: 04_ols_residuals_map.png")
    plt.close()
    
    # Moran scatterplot
    print("\n6.3 Moran Scatterplot")
    print("-" * 40)
    
    from libpysal.weights.spatial_lag import lag_spatial
    
    # Standardize residuals
    residuals_std = (residuals - residuals.mean()) / residuals.std()
    
    # Calculate spatial lag
    residuals_lag = lag_spatial(w, residuals_std.flatten())
    
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Scatter plot
    ax.scatter(residuals_std, residuals_lag, alpha=0.5, s=20)
    
    # Add regression line
    from scipy.stats import linregress
    slope, intercept, r_value, p_value, std_err = linregress(residuals_std.flatten(), residuals_lag)
    line_x = np.array([residuals_std.min(), residuals_std.max()])
    line_y = slope * line_x + intercept
    ax.plot(line_x, line_y, 'r-', linewidth=2, label=f'Slope = {slope:.3f}')
    
    # Add reference lines
    ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
    ax.axvline(x=0, color='k', linestyle='--', linewidth=0.5)
    
    # Labels and title
    ax.set_xlabel('Standardized Residuals', fontsize=12)
    ax.set_ylabel('Spatial Lag of Standardized Residuals', fontsize=12)
    ax.set_title(f"Moran's I Scatterplot\nI = {moran.I:.4f}, p = {moran.p_norm:.4f}", 
                 fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('05_moran_scatterplot.png', dpi=300, bbox_inches='tight')
    print("  Saved: 05_moran_scatterplot.png")
    plt.close()
    
    return moran

def fit_spatial_models(y, X_const, var_names, w):
    """
    Fit spatial lag and spatial error models using both ML and GMM
    """
    print("\n" + "="*80)
    print("STEP 7: ESTIMATING SPATIAL MODELS")
    print("="*80)
    
    # 7.1 Spatial Lag Model - Maximum Likelihood
    print("\n7.1 Spatial Lag Model (SAR) - Maximum Likelihood")
    print("   Estimating via ML...")
    
    lag_ml = ML_Lag(y, X_const, w=w, name_y='log_price', name_x=var_names)
    
    print("\n" + "="*60)
    print("SPATIAL LAG MODEL - ML RESULTS")
    print("="*60)
    print(lag_ml.summary)
    
    # 7.2 Spatial Lag Model - GMM (Robustness Check)
    print("\n7.2 Spatial Lag Model (SAR) - GMM (Robustness)")
    print("-" * 40)
    print("   Estimating via GMM (robust to heteroskedasticity & non-normality)...")
    
    lag_gmm = GM_Lag(y, X_const, w=w, name_y='log_price', name_x=var_names)
    
    print("\n" + "="*60)
    print("SPATIAL LAG MODEL - GMM RESULTS")
    print("="*60)
    print(lag_gmm.summary)
    
    # 7.3 Spatial Error Model - Maximum Likelihood
    print("\n7.3 Spatial Error Model (SEM) - Maximum Likelihood")
    print("-" * 40)
    print("   Estimating via ML...")
    
    error_ml = ML_Error(y, X_const, w=w, name_y='log_price', name_x=var_names)
    
    print("\n" + "="*60)
    print("SPATIAL ERROR MODEL - ML RESULTS")
    print("="*60)
    print(error_ml.summary)
    
    # 7.4 Spatial Error Model - GMM (Robustness Check)
    print("\n7.4 Spatial Error Model (SEM) - GMM (Robustness)")
    print("-" * 40)
    print("   Estimating via GMM (robust to heteroskedasticity & non-normality)...")
    
    error_gmm = GM_Error(y, X_const, w=w, name_y='log_price', name_x=var_names)
    
    print("\n" + "="*60)
    print("SPATIAL ERROR MODEL - GMM RESULTS")
    print("="*60)
    print(error_gmm.summary)
    
    return lag_ml, lag_gmm, error_ml, error_gmm

def extract_scalar(value):
    """
    Safely extract scalar value from numpy array or return value as-is
    """
    if isinstance(value, np.ndarray):
        return float(value.flatten()[0])
    return float(value)

def compare_models(ols_model, lag_ml, lag_gmm, error_ml, error_gmm):
    """
    Compare OLS vs Spatial models with correct ML vs GMM handling.
    Fully fixed drop-in version.
    """
    import numpy as np
    import pandas as pd

    print("\n" + "="*80)
    print("STEP 8: MODEL COMPARISON")
    print("="*80)


    def get_scalar(x):
        if x is None:
            return np.nan
        if isinstance(x, (float, int)):
            return float(x)
        try:
            return float(np.asarray(x).squeeze())
        except Exception:
            return np.nan


    lag_ml_rho  = get_scalar(lag_ml.rho)
    lag_gmm_rho = get_scalar(lag_gmm.rho)

    error_ml_lam  = get_scalar(error_ml.lam)
    error_gmm_lam = get_scalar(error_gmm.betas[-1])  # GM_Error stores λ here

 
    models = {
        "OLS": {
            "R² / Pseudo R²": ols_model.r2,
            "AIC": ols_model.aic,
            "Log-Likelihood": ols_model.logll,
            "Spatial Param": np.nan
        },
        "Spatial Lag (ML)": {
            "R² / Pseudo R²": lag_ml.pr2,
            "AIC": lag_ml.aic,
            "Log-Likelihood": lag_ml.logll,
            "Spatial Param": f"ρ={lag_ml_rho:.4f}"
        },
        "Spatial Lag (GMM)": {
            "R² / Pseudo R²": lag_gmm.pr2,
            "AIC": np.nan,
            "Log-Likelihood": np.nan,
            "Spatial Param": f"ρ={lag_gmm_rho:.4f}"
        },
        "Spatial Error (ML)": {
            "R² / Pseudo R²": error_ml.pr2,
            "AIC": error_ml.aic,
            "Log-Likelihood": error_ml.logll,
            "Spatial Param": f"λ={error_ml_lam:.4f}"
        },
        "Spatial Error (GMM)": {
            "R² / Pseudo R²": error_gmm.pr2,
            "AIC": np.nan,
            "Log-Likelihood": np.nan,
            "Spatial Param": f"λ={error_gmm_lam:.4f}"
        }
    }

    comparison_df = pd.DataFrame(models).T
    print("\n8.1 Model Fit Comparison")
    print("-" * 40)
    print(comparison_df.to_string())

 
    aic_df = comparison_df.dropna(subset=["AIC"])
    best_model_name = aic_df["AIC"].astype(float).idxmin()
    print(f"\n*** Best ML model by AIC: {best_model_name}")

 
    print("\n8.2 Likelihood Ratio Tests (ML)")
    print("-" * 40)

    lr_lag = 2 * (lag_ml.logll - ols_model.logll)
    print("OLS vs Spatial Lag (ML)")
    print(f"  LR statistic: {lr_lag:.4f}")
    if lr_lag > 3.841:
        print("  *** Spatial Lag significantly better than OLS")

    lr_error = 2 * (error_ml.logll - ols_model.logll)
    print("\nOLS vs Spatial Error (ML)")
    print(f"  LR statistic: {lr_error:.4f}")
    if lr_error > 3.841:
        print("  *** Spatial Error significantly better than OLS")

 
    print("\n8.3 ML vs GMM Robustness Check")
    print("-" * 40)

    print("Spatial Lag:")
    print(f"  ρ (ML):  {lag_ml_rho:.6f}")
    print(f"  ρ (GMM): {lag_gmm_rho:.6f}")
    print(f"  |Δρ|:    {abs(lag_ml_rho - lag_gmm_rho):.6f}")

    print("\nSpatial Error:")
    print(f"  λ (ML):  {error_ml_lam:.6f}")
    print(f"  λ (GMM): {error_gmm_lam:.6f}")
    print(f"  |Δλ|:    {abs(error_ml_lam - error_gmm_lam):.6f}")

    print("\nInterpretation:")
    print(" • ML models → compared via AIC / LR tests")
    print(" • GMM models → robustness under heteroskedasticity")
    print(" • Similar ML & GMM parameters → stable spatial structure")

    return comparison_df, best_model_name



def interpret_canopy_effects(lag_ml, lag_gmm, error_ml, error_gmm, X_vars):
    """
    Interpret coefficients for canopy variables across methods
    FIXED: Properly handle numpy arrays in coefficient extraction
    """
    print("\n" + "="*80)
    print("STEP 9: INTERPRETING CANOPY EFFECTS")
    print("="*80)
    
    # Get canopy variable indices
    canopy_indices = [i for i, var in enumerate(['Constant'] + X_vars) 
                      if 'canopy' in var.lower()]
    canopy_vars = [(['Constant'] + X_vars)[i] for i in canopy_indices]
    
    if len(canopy_vars) == 0:
        print("   No canopy variables found in model")
        return
    
    print("\n9.1 Canopy Coefficients Comparison (ML vs GMM)")
    print("-" * 40)
    
    # Create comparison table
    canopy_results = []
    
    for idx, var in zip(canopy_indices, canopy_vars):
        lag_ml_coef = extract_scalar(lag_ml.betas[idx])
        lag_gmm_coef = extract_scalar(lag_gmm.betas[idx])
        error_ml_coef = extract_scalar(error_ml.betas[idx])
        error_gmm_coef = extract_scalar(error_gmm.betas[idx])
        
        canopy_results.append({
            'Variable': var,
            'Lag_ML': lag_ml_coef,
            'Lag_GMM': lag_gmm_coef,
            'Error_ML': error_ml_coef,
            'Error_GMM': error_gmm_coef
        })
    
    canopy_df = pd.DataFrame(canopy_results)
    print("\n" + canopy_df.to_string(index=False))
    
    print("\n9.2 Robustness Assessment")
    print("-" * 40)
    
    for idx, var in zip(canopy_indices, canopy_vars):
        lag_ml_coef = extract_scalar(lag_ml.betas[idx])
        lag_gmm_coef = extract_scalar(lag_gmm.betas[idx])
        
        diff = abs(lag_ml_coef - lag_gmm_coef)
        pct_diff = (diff / abs(lag_ml_coef)) * 100 if lag_ml_coef != 0 else 0
        
        print(f"\n   {var}:")
        print(f"      ML:  {lag_ml_coef:.6f}")
        print(f"      GMM: {lag_gmm_coef:.6f}")
        print(f"      Difference: {diff:.6f} ({pct_diff:.1f}%)")
        
        if pct_diff < 10:
            print(f"     Estimates are robust (< 10% difference)")
        elif pct_diff < 20:
            print(f"  Moderate difference (10-20%)")
        else:
            print(f"  Large difference (> 20%) - sensitive to method choice")

# ============================================================================
# STEP 10: RESIDUAL DIAGNOSTICS
# ============================================================================

def residual_diagnostics(lag_ml, w, gdf):
    """
    Check residuals from spatial model for remaining autocorrelation
    """
    print("\n" + "="*80)
    print("STEP 10: RESIDUAL DIAGNOSTICS")
    print("="*80)
    
    residuals = lag_ml.u
    
    # Test for remaining spatial autocorrelation
    print("\n10.1 Moran's I Test on Spatial Model Residuals")
    print("-" * 40)
    
    moran_residuals = Moran(residuals.flatten(), w)
    
    print(f"   Moran's I statistic: {moran_residuals.I:.4f}")
    print(f"   Z-score: {moran_residuals.z_norm:.4f}")
    print(f"   P-value: {moran_residuals.p_norm:.6f}")
    
    if moran_residuals.p_norm > 0.05:
        print(f"\n  No significant spatial autocorrelation in residuals")
        print(f"     Spatial model adequately captures spatial dependence!")
    else:
        print(f"\n   Some spatial autocorrelation remains")
        print(f"     Consider: Spatial Durbin Model or additional spatial predictors")
    
    # Map residuals
    print("\n10.2 Mapping Spatial Model Residuals")
    print("-" * 40)
    
    gdf_plot = gdf.copy()
    gdf_plot['spatial_residuals'] = residuals
    
    fig, ax = plt.subplots(figsize=(12, 10))
    gdf_plot.plot(column='spatial_residuals', cmap='RdBu', legend=True,
                  ax=ax, markersize=2, alpha=0.6,
                  vmin=-residuals.std()*2, vmax=residuals.std()*2)
    ax.set_title('Spatial Lag Model Residuals', fontweight='bold', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('06_spatial_residuals_map.png', dpi=300, bbox_inches='tight')
    print("  Saved: 06_spatial_residuals_map.png")
    plt.close()
    
    # Histogram of residuals
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Residuals')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Spatial Model Residuals')
    axes[0].axvline(0, color='red', linestyle='--')
    
    stats.probplot(residuals.flatten(), dist="norm", plot=axes[1])
    axes[1].set_title('Q-Q Plot')
    
    plt.tight_layout()
    plt.savefig('07_spatial_residuals_dist.png', dpi=300, bbox_inches='tight')
    print("  Saved: 07_spatial_residuals_dist.png")
    plt.close()


def robustness_checks(y, X_const, var_names, gdf):
    """
    Test sensitivity to different spatial weights specifications
    """
    print("\n" + "="*80)
    print("STEP 11: ROBUSTNESS CHECKS")
    print("="*80)
    
    coords = np.array(list(zip(gdf.geometry.x, gdf.geometry.y)))
    
    print("\n11.1 Alternative Spatial Weights Matrices")
    print("-" * 40)
    
    # Test different k values
    k_values = [5, 8, 10, 15]
    results = []
    
    for k in k_values:
        print(f"\n   Testing k={k} nearest neighbors...")
        w_k = KNN.from_array(coords, k=k)
        w_k.transform = 'r'
        
        lag_k = ML_Lag(y, X_const, w=w_k, name_y='log_price', name_x=var_names)
        
        # Extract rho as scalar
        rho_k = extract_scalar(lag_k.rho)
        
        results.append({
            'k': k,
            'AIC': lag_k.aic,
            'Pseudo R²': lag_k.pr2,
            'Rho': rho_k
        })
    
    results_df = pd.DataFrame(results)
    print("\n   Results across different k values:")
    print(results_df.to_string(index=False))
    
    print("\n11.2 Distance-Based Weights (Alternative)")

    
    # Test distance thresholds
    distances = [300, 500, 800]
    for dist in distances:
        print(f"\n   Testing distance threshold={dist}m...")
        w_dist = DistanceBand.from_array(coords, threshold=dist)
        if min(w_dist.cardinalities.values()) == 0:
            print(f"  Some observations have no neighbors at {dist}m - skipping")
            continue
        w_dist.transform = 'r'
        
        lag_dist = ML_Lag(y, X_const, w=w_dist, name_y='log_price', name_x=var_names)
        print(f"      AIC: {lag_dist.aic:.2f}")
        print(f"      Avg neighbors: {np.mean(list(w_dist.cardinalities.values())):.1f}")


def export_results(ols_model, lag_ml, lag_gmm, error_ml, error_gmm, X_vars, gdf, vif_results):
    """
    Export regression tables and predictions
    FIXED: Proper ML vs GMM handling and safe scalar extraction
    """
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    print("\n" + "="*80)
    print("STEP 12: EXPORTING RESULTS")
    print("="*80)

  
    def get_scalar(x):
        if x is None:
            return np.nan
        if isinstance(x, (float, int)):
            return float(x)
        try:
            return float(np.asarray(x).squeeze())
        except Exception:
            return np.nan

 
    print("\n12.1 Creating Regression Table")
    print("-" * 40)

    var_names = ['Constant'] + X_vars
    n_vars = len(var_names)

    results_dict = {
        'Variable': var_names,
        'OLS_Coef': [get_scalar(ols_model.betas[i]) for i in range(n_vars)],
        'OLS_SE':   [np.sqrt(ols_model.vm[i, i]) for i in range(n_vars)],

        'Lag_ML_Coef': [get_scalar(lag_ml.betas[i]) for i in range(n_vars)],
        'Lag_ML_SE':   [np.sqrt(lag_ml.vm[i, i]) for i in range(n_vars)],

        'Lag_GMM_Coef': [get_scalar(lag_gmm.betas[i]) for i in range(n_vars)],
        'Lag_GMM_SE':   [np.sqrt(lag_gmm.vm[i, i]) for i in range(n_vars)],

        'Error_ML_Coef': [get_scalar(error_ml.betas[i]) for i in range(n_vars)],
        'Error_ML_SE':   [np.sqrt(error_ml.vm[i, i]) for i in range(n_vars)],

        'Error_GMM_Coef': [get_scalar(error_gmm.betas[i]) for i in range(n_vars)],
        'Error_GMM_SE':   [np.sqrt(error_gmm.vm[i, i]) for i in range(n_vars)]
    }

    results_table = pd.DataFrame(results_dict)

  
    for model in ['OLS', 'Lag_ML', 'Lag_GMM', 'Error_ML', 'Error_GMM']:
        results_table[f'{model}_tstat'] = (
            results_table[f'{model}_Coef'] / results_table[f'{model}_SE']
        )
        results_table[f'{model}_sig'] = results_table[f'{model}_tstat'].apply(
            lambda t: '***' if pd.notna(t) and abs(t) > 2.576 else
                      '**'  if pd.notna(t) and abs(t) > 1.96  else
                      '*'   if pd.notna(t) and abs(t) > 1.645 else ''
        )

    lag_ml_rho   = get_scalar(lag_ml.rho)
    lag_gmm_rho  = get_scalar(lag_gmm.rho)

    error_ml_lam  = get_scalar(error_ml.lam)
    error_gmm_lam = get_scalar(error_gmm.betas[-1])   # FIX

    spatial_params = pd.DataFrame({
        'Variable': ['Rho (ρ)', 'Lambda (λ)'],

        'OLS_Coef': [np.nan, np.nan],
        'OLS_SE':   [np.nan, np.nan],

        'Lag_ML_Coef':  [lag_ml_rho, np.nan],
        'Lag_ML_SE':    [np.sqrt(lag_ml.vm[-1, -1]), np.nan],

        'Lag_GMM_Coef': [lag_gmm_rho, np.nan],
        'Lag_GMM_SE':   [np.sqrt(lag_gmm.vm[-1, -1]), np.nan],

        'Error_ML_Coef': [np.nan, error_ml_lam],
        'Error_ML_SE':   [np.nan, np.sqrt(error_ml.vm[-1, -1])],

        'Error_GMM_Coef': [np.nan, error_gmm_lam],
        'Error_GMM_SE':   [np.nan, np.sqrt(error_gmm.vm[-1, -1])]
    })

    for model in ['Lag_ML', 'Lag_GMM', 'Error_ML', 'Error_GMM']:
        spatial_params[f'{model}_tstat'] = (
            spatial_params[f'{model}_Coef'] / spatial_params[f'{model}_SE']
        )
        spatial_params[f'{model}_sig'] = spatial_params[f'{model}_tstat'].apply(
            lambda t: '***' if pd.notna(t) and abs(t) > 2.576 else
                      '**'  if pd.notna(t) and abs(t) > 1.96  else
                      '*'   if pd.notna(t) and abs(t) > 1.645 else ''
        )

    spatial_params['OLS_tstat'] = [np.nan, np.nan]
    spatial_params['OLS_sig'] = ['', '']

    results_table = pd.concat([results_table, spatial_params], ignore_index=True)

    results_table.to_csv('regression_results_table.csv', index=False)
    print("  Saved: regression_results_table.csv")

    vif_results.to_csv('vif_diagnostics.csv', index=False)
    print("  Saved: vif_diagnostics.csv")

    print("\n   Key Variables Summary (Canopy + Spatial Params):")
    display_cols = ['Variable', 'Lag_ML_Coef', 'Lag_ML_sig', 'Lag_GMM_Coef', 'Lag_GMM_sig']
    canopy_rows = results_table[results_table['Variable'].str.contains('canopy|Rho|Lambda', na=False)]
    print(canopy_rows[display_cols].to_string(index=False))

  
    print("\n12.2 Exporting Predictions")
    print("-" * 40)

    gdf_export = gdf.copy()
    gdf_export['predicted_log_price'] = lag_ml.predy.flatten()
    gdf_export['predicted_price'] = np.exp(lag_ml.predy.flatten())
    gdf_export['residual'] = lag_ml.u.flatten()

    try:
        gdf_export.to_file('property_predictions.gpkg', driver='GPKG')
        print("  Saved: property_predictions.gpkg")
    except Exception:
        gdf_export.to_file('property_predictions.shp')
        print("  Saved: property_predictions.shp")

    export_cols = ['log_price', 'GrossSalePrice', 'predicted_log_price',
                   'predicted_price', 'residual'] + X_vars
    export_cols = [c for c in export_cols if c in gdf_export.columns]
    gdf_export[export_cols].to_csv('property_predictions.csv', index=False)
    print("  Saved: property_predictions.csv")

  
    print("\n12.3 Prediction Quality")
    print("-" * 40)

    mae = np.mean(np.abs(gdf_export['residual']))
    rmse = np.sqrt(np.mean(gdf_export['residual'] ** 2))

    print(f"   MAE:  {mae:.4f}")
    print(f"   RMSE: {rmse:.4f}")
    print(f"   Pseudo R² (ML):  {lag_ml.pr2:.4f}")
    print(f"   Pseudo R² (GMM): {lag_gmm.pr2:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].scatter(gdf_export['log_price'], gdf_export['predicted_log_price'],
                    alpha=0.3, s=10)
    axes[0].plot(
        [gdf_export['log_price'].min(), gdf_export['log_price'].max()],
        [gdf_export['log_price'].min(), gdf_export['log_price'].max()],
        'r--', linewidth=2
    )
    axes[0].set_title('Actual vs Predicted (Log Price)')
    axes[0].set_xlabel('Actual')
    axes[0].set_ylabel('Predicted')
    axes[0].grid(True, alpha=0.3)

    axes[1].scatter(gdf_export['predicted_log_price'], gdf_export['residual'],
                    alpha=0.3, s=10)
    axes[1].axhline(0, color='r', linestyle='--', linewidth=2)
    axes[1].set_title('Residuals vs Predicted')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Residual')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('08_prediction_diagnostics.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("  Saved: 08_prediction_diagnostics.png")



def main(filepath):
    """
    Run complete spatial hedonic analysis pipeline
    """
    print("\n")
    print("="*80)
    print(" SPATIAL HEDONIC PRICE ANALYSIS: STREET TREE CANOPY EFFECTS")
    print(" WITH VIF DIAGNOSTICS & GMM ROBUSTNESS CHECKS")
    print("="*80)
    print("\n")
    
    # Step 1: Load data
    gdf = load_and_prepare_data(filepath)
    
    # Step 2: EDA
    key_vars, canopy_vars, imd_vars = exploratory_analysis(gdf)
    
    # Step 3: Feature engineering
    gdf_model = feature_engineering(gdf, imd_vars)
    
    # Step 4: Baseline OLS + VIF test
    ols_model, ols_residuals, X_vars, vif_results = fit_baseline_ols(gdf_model)
    
    # Step 5: Spatial weights
    w = create_spatial_weights(gdf_model, method='knn', k=8)
    
    # Step 6: Test spatial autocorrelation
    moran = test_spatial_autocorrelation(ols_residuals, w, gdf_model)
    
    # Prepare data for spatial models
    y = gdf_model['log_price'].values.reshape(-1, 1)
    X = gdf_model[X_vars].values
    X_const = np.hstack([np.ones((X.shape[0], 1)), X])
    var_names = ['Constant'] + X_vars
    
    # Step 7: Fit spatial models (ML + GMM)
    lag_ml, lag_gmm, error_ml, error_gmm = fit_spatial_models(y, X_const, var_names, w)
    
    # Step 8: Model comparison
    comparison_df, best_model = compare_models(ols_model, lag_ml, lag_gmm, error_ml, error_gmm)
    
    # Step 9: Interpret canopy effects
    interpret_canopy_effects(lag_ml, lag_gmm, error_ml, error_gmm, X_vars)
    
    # Step 10: Residual diagnostics
    residual_diagnostics(lag_ml, w, gdf_model)
    
    # Step 11: Robustness checks
    robustness_checks(y, X_const, var_names, gdf_model)
    
    # Step 12: Export results
    export_results(ols_model, lag_ml, lag_gmm, error_ml, error_gmm, X_vars, gdf_model, vif_results)



if __name__ == "__main__":
    # Your actual file path
    filepath = "../output/property_all.gpkg"
    
    # Run the complete pipeline
    main(filepath)
    
    



 SPATIAL HEDONIC PRICE ANALYSIS: STREET TREE CANOPY EFFECTS
 WITH VIF DIAGNOSTICS & GMM ROBUSTNESS CHECKS


STEP 1: LOADING AND PREPARING DATA

   Loading data from: ../output/property_all.gpkg
   ✓ Loaded 12,317 properties
   ✓ CRS: EPSG:2193
   ✓ Columns: 34
   ✓ All required columns present

STEP 2: EXPLORATORY DATA ANALYSIS

2.1 Summary Statistics
----------------------------------------
          log_price  GrossSalePrice     AgeAtSale      LandArea  \
count  12317.000000    1.231700e+04  12317.000000  12317.000000   
mean      13.150489    5.465392e+05     49.211496    681.107494   
std        0.335514    2.145990e+05     30.921931   1368.324650   
min       11.608236    1.100000e+05      2.000000     53.000000   
25%       12.928779    4.120000e+05     22.000000    501.000000   
50%       13.108264    4.930000e+05     52.000000    625.000000   
75%       13.340695    6.220000e+05     72.000000    746.000000   
max       14.357835    1.720000e+06    144.000000  50788.000000   


In [10]:
gdf = gpd.read_file('../output/property_all.gpkg')

In [11]:
gdf.columns

Index(['GrossSalePrice', 'AgeAtSale', 'LandArea', 'TotalFloorArea',
       'water_DIST', 'bus_DIST', 'Census_Pop', 'RnkIMDNoEm', 'RnkIMDNoIn',
       'RnkIMDNoCr', 'RnkIMDNoHo', 'RnkIMDNoHe', 'RnkIMDNoEd', 'RnkIMDNoAc',
       'DECILE_high', 'DECILE_prim', 'Median_Income', 'CBD_DIST',
       'cycleways_DIST', 'cycle_DENS', 'year_2018', 'year_2019', 'canopy_0_25',
       'canopy_25_50', 'canopy_50_75', 'canopy_75_100', 'canopy_100_150',
       'canopy_150_200', 'canopy_200_250', 'canopy_250_300', 'canopy_300_350',
       'canopy_350_400', 'log_price', 'geometry'],
      dtype='object')